In [2]:
from google.colab import drive
drive.mount('/content/drive')

# 设置你的数据目录
base_dir = '/content/drive/MyDrive/Colab Notebooks/ant_data'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install lightgbm scikit-learn matplotlib pandas numpy


In [ ]:
!pip install --upgrade lightgbm


In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
import os
import zipfile
import matplotlib.font_manager as fm

# LightGBM 模型作用：特征重要性解释 ✅ 非常清晰，可视化强
# 快速原型验证        ✅ 训练快，调参简单
# 回归任务（如收益率预测）  ✅ 表现稳定，误差可控
# 排序任务（如选股打分）   ✅ 适合做打分器或信号筛选器
# ✅ 配置matplotlib支持中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']  # 或者其他支持中文的字体
plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题

# ✅ 参数设置
stock_codes = ['601138','002269']  # 可扩展多个股票
start_date = "20210924"
end_date = datetime.today().strftime("%Y%m%d")
save_path = "./data/"
chart_dir = "./charts"
os.makedirs(save_path, exist_ok=True)
os.makedirs(chart_dir, exist_ok=True)
today_str = datetime.today().strftime("%Y-%m-%d")

# ✅ 标准化列名函数
def standardize_columns(df):
    column_mapping = {
        "日期": "Date", "股票代码": "Code", "开盘": "Open", "收盘": "Close",
        "最高": "High", "最低": "Low", "成交量": "Volume", "成交额": "Amount",
        "振幅": "Amplitude", "涨跌幅": "ChangePct", "涨跌额": "ChangeAmt", "换手率": "Turnover",
        "opendate": "Date", "netamount": "MainNetInflow", "ratioamount": "MainInflowRatio"
    }
    df.rename(columns=column_mapping, inplace=True)
    return df

# ✅ RSI计算函数
def compute_rsi(series, window=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# ✅ 数据加载与特征构造
def load_and_prepare_data(code, backtest_start, backtest_end):
    prefix = code[-6:]
    market_prefix = "sz" if code.startswith("0") else "sh"
    daily_path = f"/content/drive/MyDrive/Colab Notebooks/ant_data/kline/{prefix}_daily.csv"
    fundflow_path = f"/content/drive/MyDrive/Colab Notebooks/ant_data/jiqi_do/{market_prefix}{code}.csv"

    if not os.path.exists(daily_path):
        print(f"⚠️ 缺失数据文件：{daily_path}")
        return None, None, None

    df = pd.read_csv(daily_path)
    df["Date"] = pd.to_datetime(df["Date"])
    df.sort_values("Date", inplace=True)
    df.set_index("Date", inplace=True)

    if os.path.exists(fundflow_path):
        df_fund = pd.read_csv(fundflow_path)
        df_fund = standardize_columns(df_fund)
        df_fund["Date"] = pd.to_datetime(df_fund["Date"])
        df_fund.set_index("Date", inplace=True)
        df = df.merge(df_fund[["MainNetInflow", "MainInflowRatio"]], left_index=True, right_index=True, how="left")

    # 技术指标构造
    df["MA5"] = df["Close"].rolling(window=5).mean()
    df["MA10"] = df["Close"].rolling(window=10).mean()
    df["EMA12"] = df["Close"].ewm(span=12, adjust=False).mean()
    df["EMA26"] = df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD"] = df["EMA12"] - df["EMA26"]
    df["RSI"] = compute_rsi(df["Close"], window=14)
    df["Return_1d"] = df["Close"].pct_change()
    df["Volatility_5d"] = df["Close"].rolling(window=5).std()
    df["Weekday"] = df.index.weekday
    df["IsMonthEnd"] = df.index.is_month_end.astype(int)

    # 资金流衍生特征
    if "MainNetInflow" in df.columns and "MainInflowRatio" in df.columns:
        df["MainNetInflow_Smooth"] = df["MainNetInflow"].ewm(span=5, adjust=False).mean()
        df["MainNetInflow_Normalized"] = (df["MainNetInflow"] - df["MainNetInflow"].mean()) / df["MainNetInflow"].std()
        # Use .fillna(0) or .dropna() as pct_change can produce NaNs
        df["MainNetInflow_Ratio_Change"] = df["MainInflowRatio"].pct_change().fillna(0)
        df["MainNetInflow_5d_Std"] = df["MainNetInflow"].rolling(window=5).std()
    else:
        # Initialize these columns with 0 if main funds flow data is missing
        df["MainNetInflow_Smooth"] = 0
        df["MainNetInflow_Normalized"] = 0
        df["MainNetInflow_Ratio_Change"] = 0
        df["MainNetInflow_5d_Std"] = 0
        print("⚠️ Funds flow columns not found, initializing derivative features to 0.")


    # 异动信号（涨停/跌停/资金突变）
    # 涨停标记：假设涨停是接近最高价且涨幅较大
    df["IsLimitUp"] = ((df["High"] >= df["Close"] * 0.99) & (df["ChangePct"] > 9.5)).astype(int)
    # 跌停标记：假设跌停是接近最低价且跌幅较大
    df["IsLimitDown"] = ((df["Low"] <= df["Close"] * 1.01) & (df["ChangePct"] < -9.5)).astype(int)

    # 连续涨停次数 (Simplified: count consecutive IsLimitUp)
    # This requires a more sophisticated approach for true consecutive counting,
    # but a simple rolling sum can indicate recent frequency.
    df["ConsecutiveLimitUp_5d"] = df["IsLimitUp"].rolling(window=5).sum()

    # 主力资金流入突变：资金净流入 > 均值 + 2σ (requires MainNetInflow)
    if "MainNetInflow" in df.columns:
        mean_inflow = df["MainNetInflow"].mean()
        std_inflow = df["MainNetInflow"].std()
        df["IsAbnormalInflow"] = (df["MainNetInflow"] > mean_inflow + 2 * std_inflow).astype(int)
    else:
        df["IsAbnormalInflow"] = 0
        print("⚠️ MainNetInflow column not found, skipping IsAbnormalInflow feature.")

    # 换手率激增：换手率 > 历史均值 + 3σ
    if "Turnover" in df.columns:
        mean_turnover = df["Turnover"].mean()
        std_turnover = df["Turnover"].std()
        df["IsHighTurnover"] = (df["Turnover"] > mean_turnover + 3 * std_turnover).astype(int)
    else:
        df["IsHighTurnover"] = 0
        print("⚠️ Turnover column not found, skipping IsHighTurnover feature.")


    # 构造标签
    df["Label"] = (df["Close"].shift(-1) > df["Close"]).astype(int)
    df.dropna(inplace=True)

    # 特征列表
    features = [
        "Open", "High", "Low", "Close", "Volume", "Amount", "Turnover",
        "MA5", "MA10", "EMA12", "EMA26", "MACD", "RSI",
        "Return_1d", "Volatility_5d", "Weekday", "IsMonthEnd",
        "MainNetInflow", "MainInflowRatio", "ChangePct",
        "MainNetInflow_Smooth", "MainNetInflow_Normalized",
        "MainNetInflow_Ratio_Change", "MainNetInflow_5d_Std",
        "IsLimitUp", "IsLimitDown", "ConsecutiveLimitUp_5d",
        "IsAbnormalInflow", "IsHighTurnover" # Add new features to the list
    ]

    return df, features

# ✅ 主流程：多股票分类训练 + 回测 + 特征图
for code in stock_codes:
    df, features = load_and_prepare_data(code, start_date, end_date)
    if df is None:
        continue



    # 划分训练集和回测集
    backtest_days = 10
    X_all = df[features]
    y_all = df["Label"]

    X_train_full = X_all.iloc[:-backtest_days]
    y_train_full = y_all.iloc[:-backtest_days]
    X_test = X_all.iloc[-backtest_days:]
    y_test = y_all.iloc[-backtest_days:]
    date_test = df.index[-backtest_days:]

    # ✅ 从训练集中划分验证集
    X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1, shuffle=False)

    # ✅ 训练 LightGBM 模型（带验证集和早停）
    model = lgb.LGBMClassifier(n_estimators=500, max_depth=6)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='binary_logloss',
        callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=50)]
    )

    # ✅ 使用最优轮次预测
    y_pred = model.predict(X_test, num_iteration=model.best_iteration_)

    # ✅ 保存模型
    model_dir = "./models"
    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, f"{code}_lightgbm_{today_str}.txt")
    model.booster_.save_model(model_path)
    print(f"✅ 模型已保存：{model_path}")


    # 预测与评估
    # y_pred = model.predict(X_test)
    print(f"\n📊 股票 {code} 分类报告：")
    print(classification_report(y_test, y_pred))
    print("📉 混淆矩阵：")
    print(confusion_matrix(y_test, y_pred))

    # 绘图：回测预测效果
    plt.figure(figsize=(10, 4))
    plt.plot(date_test, y_test.values, label="真实涨跌", marker='o', color='blue')
    plt.plot(date_test, y_pred, label="预测涨跌", marker='x', color='orange')
    plt.title(f"{code} - 最近10天涨跌分类预测效果")
    plt.xlabel("日期")
    plt.ylabel("涨跌标签（1=涨，0=跌）")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    chart_path = os.path.join(chart_dir, f"{code}_backtest_{today_str}.png")
    plt.savefig(chart_path)
    plt.close()
    print(f"✅ 回测图表已保存：{chart_path}")
    # ✅ 柱状图：真实涨跌幅分类 vs 模型预测
    fig, ax = plt.subplots(figsize=(12, 5))

    # 获取真实涨跌幅
    change_pct = df["ChangePct"].iloc[-backtest_days:]
    labels_true = (change_pct > 0).astype(int)  # 真实标签：涨=1，跌=0

    # 设置颜色：真实涨为红，跌为绿
    bar_colors = ['red' if label == 1 else 'green' for label in labels_true]

    # 绘制真实涨跌柱状图
    ax.bar(date_test, change_pct, color=bar_colors, width=0.6, label="真实涨跌幅")

    # 在柱子上叠加模型预测点（红=预测涨，绿=预测跌）
    pred_colors = ['red' if pred == 1 else 'green' for pred in y_pred]
    ax.scatter(date_test, [0]*backtest_days, color=pred_colors, marker='x', s=80, label="模型预测")

    # 图表美化
    ax.set_title(f"{code} - 真实涨跌幅 vs 模型预测（柱状图）")
    ax.set_ylabel("涨跌幅 (%)")
    ax.axhline(0, color='gray', linestyle='--')
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()

    # 保存图表
    bar_chart_path = os.path.join(chart_dir, f"{code}_bar_vs_pred_{today_str}.png")
    plt.savefig(bar_chart_path)
    plt.close()
    print(f"✅ 柱状图已保存：{bar_chart_path}")


    # 特征重要性图
    plt.figure(figsize=(10, 6))
    lgb.plot_importance(model, max_num_features=15, importance_type='gain')
    plt.title(f"{code} - LightGBM 特征重要性（按增益）")
    plt.tight_layout()
    importance_path = os.path.join(chart_dir, f"{code}_feature_importance_{today_str}.png")
    plt.savefig(importance_path)
    plt.close()
    print(f"✅ 特征重要性图已保存：{importance_path}")


def zip_today_charts(chart_dir="./charts", zip_name="charts_output.zip"):
    today_str = datetime.today().strftime("%Y-%m-%d")
    with zipfile.ZipFile(zip_name, 'w') as zipf:
        for file in os.listdir(chart_dir):
            if file.endswith(".png") and today_str in file:
                zipf.write(os.path.join(chart_dir, file), arcname=file)
    print(f"📦 所有图表已打包为：{zip_name}")
    # ✅ 打包图表
zip_name = f"charts_{today_str}.zip"

/tmp/ipython-input-1714142021.py:91: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["MainNetInflow_Ratio_Change"] = df["MainInflowRatio"].pct_change().fillna(0)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr

[LightGBM] [Info] Number of positive: 393, number of negative: 454
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000500 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5487
[LightGBM] [Info] Number of data points in the train set: 847, number of used features: 26
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.463991 -> initscore=-0.144288
[LightGBM] [Info] Start training from score -0.144288
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 15 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [

/tmp/ipython-input-1714142021.py:211: UserWarning: Glyph 26085 (\N{CJK UNIFIED IDEOGRAPH-65E5}) missing from font(s) DejaVu Sans.
  plt.savefig(chart_path)
/tmp/ipython-input-1714142021.py:211: UserWarning: Glyph 26399 (\N{CJK UNIFIED IDEOGRAPH-671F}) missing from font(s) DejaVu Sans.
  plt.savefig(chart_path)
/tmp/ipython-input-1714142021.py:211: UserWarning: Glyph 28072 (\N{CJK UNIFIED IDEOGRAPH-6DA8}) missing from font(s) DejaVu Sans.
  plt.savefig(chart_path)
/tmp/ipython-input-1714142021.py:211: UserWarning: Glyph 36300 (\N{CJK UNIFIED IDEOGRAPH-8DCC}) missing from font(s) DejaVu Sans.
  plt.savefig(chart_path)
/tmp/ipython-input-1714142021.py:211: UserWarning: Glyph 26631 (\N{CJK UNIFIED IDEOGRAPH-6807}) missing from font(s) DejaVu Sans.
  plt.savefig(chart_path)
/tmp/ipython-input-1714142021.py:211: UserWarning: Glyph 31614 (\N{CJK UNIFIED IDEOGRAPH-7B7E}) missing from font(s) DejaVu Sans.
  plt.savefig(chart_path)
/tmp/ipython-input-1714142021.py:211: UserWarning: Glyph 65288 (

✅ 回测图表已保存：./charts/601138_backtest_2025-10-10.png


/tmp/ipython-input-1714142021.py:241: UserWarning: Glyph 28072 (\N{CJK UNIFIED IDEOGRAPH-6DA8}) missing from font(s) DejaVu Sans.
  plt.savefig(bar_chart_path)
/tmp/ipython-input-1714142021.py:241: UserWarning: Glyph 36300 (\N{CJK UNIFIED IDEOGRAPH-8DCC}) missing from font(s) DejaVu Sans.
  plt.savefig(bar_chart_path)
/tmp/ipython-input-1714142021.py:241: UserWarning: Glyph 24133 (\N{CJK UNIFIED IDEOGRAPH-5E45}) missing from font(s) DejaVu Sans.
  plt.savefig(bar_chart_path)
/tmp/ipython-input-1714142021.py:241: UserWarning: Glyph 30495 (\N{CJK UNIFIED IDEOGRAPH-771F}) missing from font(s) DejaVu Sans.
  plt.savefig(bar_chart_path)
/tmp/ipython-input-1714142021.py:241: UserWarning: Glyph 23454 (\N{CJK UNIFIED IDEOGRAPH-5B9E}) missing from font(s) DejaVu Sans.
  plt.savefig(bar_chart_path)
/tmp/ipython-input-1714142021.py:241: UserWarning: Glyph 27169 (\N{CJK UNIFIED IDEOGRAPH-6A21}) missing from font(s) DejaVu Sans.
  plt.savefig(bar_chart_path)
/tmp/ipython-input-1714142021.py:241: Us

...

✅ 柱状图已保存：./charts/002269_bar_vs_pred_2025-10-10.png


/tmp/ipython-input-1714142021.py:252: UserWarning: Glyph 29305 (\N{CJK UNIFIED IDEOGRAPH-7279}) missing from font(s) DejaVu Sans.
  plt.savefig(importance_path)
/tmp/ipython-input-1714142021.py:252: UserWarning: Glyph 24449 (\N{CJK UNIFIED IDEOGRAPH-5F81}) missing from font(s) DejaVu Sans.
  plt.savefig(importance_path)
/tmp/ipython-input-1714142021.py:252: UserWarning: Glyph 37325 (\N{CJK UNIFIED IDEOGRAPH-91CD}) missing from font(s) DejaVu Sans.
  plt.savefig(importance_path)
/tmp/ipython-input-1714142021.py:252: UserWarning: Glyph 35201 (\N{CJK UNIFIED IDEOGRAPH-8981}) missing from font(s) DejaVu Sans.
  plt.savefig(importance_path)
/tmp/ipython-input-1714142021.py:252: UserWarning: Glyph 24615 (\N{CJK UNIFIED IDEOGRAPH-6027}) missing from font(s) DejaVu Sans.
  plt.savefig(importance_path)
/tmp/ipython-input-1714142021.py:252: UserWarning: Glyph 65288 (\N{FULLWIDTH LEFT PARENTHESIS}) missing from font(s) DejaVu Sans.
  plt.savefig(importance_path)
/tmp/ipython-input-1714142021.py:2

✅ 特征重要性图已保存：./charts/002269_feature_importance_2025-10-10.png


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>